# PreTest 01: Binary Classification แบบครบวงจร — การพยากรณ์ฝนตกในวันถัดไป
### จากข้อมูลดิบ → ทำความสะอาด → Visualization → เตรียมข้อมูล → เทรน 3 โมเดล → วัดผล → เลือกโมเดล

> **ที่มา:** ดัดแปลงและขยายจาก Kaggle Notebook
> [Logistic Regression Classifier Tutorial](https://www.kaggle.com/code/prashant111/logistic-regression-classifier-tutorial) โดย Prashant Banerjee
> สิ่งที่เพิ่ม/แก้: คำอธิบายภาษาไทยทุกขั้นตอน, ลบแถวที่ Target หาย, แก้การอ่าน Confusion Matrix ให้ถูกต้อง,
> คำนวณขอบ Outlier จาก Train Set เท่านั้น, เพิ่มโมเดล **Decision Tree** และ **Random Forest**, ตารางเปรียบเทียบ 3 โมเดล
>
> **ชุดข้อมูล (Dataset):** [Rain in Australia](https://www.kaggle.com/datasets/jsphyg/weather-dataset-rattle-package) — ข้อมูลอากาศรายวัน ~10 ปี จาก 49 เมืองในออสเตรเลีย

---

## ภาพรวมขั้นตอน (สามารถใช้เป็นโครงสร้างในการทำข้อสอบได้)

| ขั้น | งาน | คำถามที่ต้องตอบ |
|---|---|---|
| 1 | กำหนดโจทย์ | ทำนายอะไร? เป็น Classification หรือ Regression? วัดผลด้วยอะไร? |
| 2 | โหลดข้อมูล | ข้อมูลมีกี่แถว กี่คอลัมน์? |
| 3 | สำรวจข้อมูลเบื้องต้น | แต่ละคอลัมน์คืออะไร ชนิดข้อมูลถูกหรือไม่ |
| 4 | ทำความสะอาด (Data Cleaning) | Target หายหรือไม่ ซ้ำหรือไม่ ค่าหายเท่าใด |
| 5 | EDA & Visualization | Target สมดุลหรือไม่ มี Outlier หรือไม่ ตัวแปรไหนสัมพันธ์กับ Target? |
| 6 | เตรียมข้อมูล (Preprocessing) | Split → จัดการ Outlier → เติมค่าหาย → Encode → Scale |
| 7–9 | เทรนโมเดล 3 ตัว | Logistic Regression, Decision Tree, Random Forest |
| 10 | เปรียบเทียบและเลือกโมเดล | ตัวไหนดีที่สุด เพราะอะไร? |

> **หลักการสำคัญ:** ทุกอย่างที่ "เรียนรู้จากข้อมูล" (ค่า median, mode, ขอบ outlier, min/max ของ scaler)
> ต้องคำนวณจาก **Train Set เท่านั้น** แล้วค่อยนำไปใช้กับ Test Set (มิฉะนั้นจะเกิด Data Leakage)

---
# ขั้นที่ 1: กำหนดโจทย์ (Problem Definition)

- **เป้าหมาย:** ใช้ข้อมูลอากาศ **วันนี้** ทำนายว่า **พรุ่งนี้ฝนจะตกหรือไม่** (`RainTomorrow` = Yes / No)
- **ประเภทปัญหา:** Target มีเพียง 2 ค่า → **Binary Classification**
- **วัดผลด้วย:** Accuracy, Precision, Recall, F1-score, ROC-AUC (จะอธิบายทีละตัวในขั้นที่ 7)

---
# ขั้นที่ 2: นำเข้าไลบรารีและโหลดข้อมูล

โค้ดด้านล่างจะหาไฟล์ในโฟลเดอร์ `Data/` ก่อน ถ้าไม่มี (เช่นเปิดบน Google Colab) จะดาวน์โหลดจาก Kaggle ให้อัตโนมัติด้วย `kagglehub`

---
# ขั้นที่ 3: สำรวจข้อมูลเบื้องต้น (First Look)
### 3.1 ดูตัวอย่างข้อมูล 5 แถวแรก

### 3.2 ความหมายของแต่ละคอลัมน์ (Data Dictionary)

| คอลัมน์ | ความหมาย | ชนิด |
|---|---|---|
| `Date` | วันที่ | ข้อความ (ต้องแปลง) |
| `Location` | เมืองที่วัด (49 เมือง) | Categorical |
| `MinTemp`, `MaxTemp` | อุณหภูมิต่ำสุด/สูงสุด (°C) | Numerical |
| `Rainfall` | ปริมาณฝนวันนี้ (มม.) | Numerical |
| `Evaporation` | ปริมาณการระเหย (มม.) | Numerical |
| `Sunshine` | จำนวนชั่วโมงแดด | Numerical |
| `WindGustDir`, `WindGustSpeed` | ทิศ/ความเร็วลมกระโชกแรงสุด | Cat / Num |
| `WindDir9am`, `WindDir3pm` | ทิศลมตอน 9 โมง / บ่าย 3 | Categorical |
| `WindSpeed9am`, `WindSpeed3pm` | ความเร็วลม (กม./ชม.) | Numerical |
| `Humidity9am`, `Humidity3pm` | ความชื้น (%) | Numerical |
| `Pressure9am`, `Pressure3pm` | ความกดอากาศ (hPa) | Numerical |
| `Cloud9am`, `Cloud3pm` | ปริมาณเมฆ (0–8 ส่วน) | Numerical |
| `Temp9am`, `Temp3pm` | อุณหภูมิ ณ เวลานั้น (°C) | Numerical |
| `RainToday` | วันนี้ฝนตกหรือไม่ (ฝน > 1 มม.) | Categorical (Yes/No) |
| **`RainTomorrow`** | **พรุ่งนี้ฝนตกหรือไม่ ← Target** | Categorical (Yes/No) |

### 3.3 ชนิดข้อมูลและจำนวนค่าที่ไม่หาย (`info`)

> **ข้อสังเกต:** `Non-Null Count` ของหลายคอลัมน์น้อยกว่า 145,460 แปลว่า **มีค่าหาย (Missing) จำนวนมาก**
> เช่น `Sunshine`, `Evaporation`, `Cloud9am` หายเกือบครึ่ง

### 3.4 สถิติสรุปของตัวแปรตัวเลข (`describe`)

> **ข้อสังเกต:** ดู `Rainfall`: ค่า 75% (Q3) = 0.8 มม. แต่ max = 371 มม. → การกระจาย **เบ้ขวามาก** และน่าจะมี **Outlier**
> (`WindSpeed9am`, `Evaporation` ก็มีลักษณะคล้ายกัน) จะตรวจละเอียดในขั้นที่ 5

---
# ขั้นที่ 4: ทำความสะอาดข้อมูล (Data Cleaning)

### 4.1 ลบแถวที่ไม่มีคำตอบ (Target หาย)
แถวที่ไม่รู้ว่าพรุ่งนี้ฝนตกหรือไม่ **ใช้สอนโมเดลไม่ได้** และ **ห้ามประมาณค่า Target ขึ้นเอง** (เพราะจะเป็นการสร้างคำตอบปลอมขึ้นมาเอง)

### 4.2 ตรวจข้อมูลซ้ำ (Duplicates)

> **ผลลัพธ์:** ไม่มีแถวซ้ำ (ถ้ามี ใช้ `df = df.drop_duplicates()`)

### 4.3 แยกตัวแปรเป็น Categorical และ Numerical
ทั้งสองแบบต้องจัดการต่างกัน (เติมค่าหายต่างวิธี, Encode เฉพาะ Categorical, Scale เฉพาะ Numerical)

### 4.4 สำรวจค่าที่หาย (Missing Values)

> **ตัดสินใจ:** แม้ `Sunshine` จะหายเกือบ 50% แต่เราจะ **ไม่ลบคอลัมน์** และจะเติมด้วยค่า median ในขั้นที่ 6
> (ทำหลังแบ่ง Train/Test เพื่อป้องกัน Data Leakage) — ในงานจริงอาจทดลองทั้งแบบลบและไม่ลบ แล้วเทียบผล

### 4.5 จำนวนหมวดหมู่ในแต่ละตัวแปร Categorical (Cardinality)
หากมีจำนวนหมวดหมู่มาก การทำ One-Hot Encoding จะสร้างคอลัมน์จำนวนมาก

> **ข้อสังเกต:** `Date` มีกว่า 3,000 ค่า (เพราะเป็นวันที่) → **ไม่ควรทำ One-Hot Encoding โดยตรง** ต้องแยกเป็นปี/เดือน/วันก่อน

### 4.6 Feature Engineering: แยกวันที่เป็น ปี / เดือน / วัน
`Month` น่าจะมีประโยชน์ เพราะฤดูกาลมีผลต่อฝน

---
# ขั้นที่ 5: EDA & Visualization

### 5.1 ความสมดุลของ Target (Class Balance)

> **ข้อมูลไม่สมดุล (Imbalanced):** วันฝนไม่ตก ~78% ฝนตก ~22%
> หากทำนายว่า "ไม่ตก" ทุกวัน จะได้ Accuracy ≈ 78% ทันทีโดยไม่ต้องมีโมเดล → เรียกว่า **Null Accuracy**
> ดังนั้นโมเดลต้องได้ **มากกว่า 78%** ถึงจะมีประโยชน์ และต้องดู Precision/Recall ประกอบด้วย

### 5.2 ตรวจ Outlier ด้วย Boxplot และ Histogram

> **ข้อสังเกต:** ทั้ง 4 ตัวมีจุดนอกกล่องจำนวนมาก และ Histogram เบ้ขวา → ต้องจัดการ Outlier

### 5.3 คำนวณขอบเขต Outlier ด้วย IQR
สูตร: `IQR = Q3 − Q1`, ขอบบน = `Q3 + k × IQR`
- ปกติ `k = 1.5` แต่ข้อมูลนี้เบ้มาก จึงใช้ `k = 3` (Extreme outlier) ตามต้นฉบับ เพื่อไม่ตัดข้อมูลมากเกินไป

> **หมายเหตุ:** ตรงนี้คำนวณจากข้อมูลทั้งหมดเพื่อ **ดูภาพรวม** เท่านั้น ตอนจัดการจริง (ขั้นที่ 6) จะคำนวณขอบใหม่จาก **Train Set**

### 5.4 ความสัมพันธ์ระหว่างตัวแปรกับฝนในวันถัดไป

> **ข้อสังเกต:** วันก่อนฝนตก: **ความชื้นตอนบ่ายสูงกว่า**, **แดดน้อยกว่า**, **ความกดอากาศต่ำกว่า** → ตัวแปรเหล่านี้น่าจะเป็น Feature ที่ดี

### 5.5 ความสัมพันธ์ระหว่างฝนวันนี้กับฝนวันถัดไป (Categorical × Categorical)

> **ข้อสังเกต:** ถ้าวันนี้ฝนตก โอกาสพรุ่งนี้ตกสูงขึ้นมาก (~46% เทียบกับ ~15%) และเดือนกลางปี (ฤดูหนาวของออสเตรเลีย) ฝนตกบ่อยกว่า

### 5.6 Correlation ระหว่างตัวแปรตัวเลข

> **ข้อสังเกต:** คู่ที่สัมพันธ์กันสูงมาก เช่น `MaxTemp`–`Temp3pm`, `Pressure9am`–`Pressure3pm` (≈ 0.9+) แปลว่าข้อมูลซ้ำซ้อนกัน
> (Multicollinearity) — ไม่ทำให้ Tree เสีย แต่ทำให้ค่า coefficient ของ Logistic Regression ตีความยากขึ้น

---
# ขั้นที่ 6: เตรียมข้อมูลสำหรับโมเดล (Preprocessing)

### 6.1 แยก X (Feature) และ y (Target)
แปลง Target เป็นตัวเลข: `Yes → 1` (ฝนตก = คลาสที่สนใจ / Positive), `No → 0`

### 6.2 แบ่ง Train / Test (80 / 20)
- `stratify=y` → ให้สัดส่วนฝนตก/ไม่ตกใน Train และ Test เท่ากัน (สำคัญมากกับข้อมูลไม่สมดุล)
- `random_state=0` → สุ่มแบบเดิมทุกครั้ง ผลลัพธ์จะเหมือนเดิม

### 6.3 จัดการ Outlier ด้วยการ "ตัดเพดาน" (Capping / Top-coding)
แทนที่จะลบแถวทิ้ง (เสียข้อมูล) เราจะ **แทนค่าที่เกินขอบบนด้วยค่าขอบบน** ขอบบนคำนวณจาก Train Set

> **ทำไมต้องทำก่อนเติมค่าหาย?** ถ้าเติม median ก่อน คอลัมน์ที่มีค่าหายจำนวนมาก (เช่น `Evaporation` หาย ~43%)
> จะมีค่า median ซ้ำกันจำนวนมาก ทำให้ IQR แคบผิดปกติ และตัดเพดานต่ำเกินไป (`quantile()` ข้ามค่า NaN ให้อยู่แล้ว)

### 6.4 เติมค่าหาย (Imputation) ด้วยค่าจาก Train Set
| ชนิด | วิธี | เหตุผล |
|---|---|---|
| Numerical | **Median** | ทนต่อ Outlier มากกว่า Mean (ข้อมูลชุดนี้มี Outlier จำนวนมาก) |
| Categorical | **Mode** (ค่าที่พบบ่อยที่สุด) | ค่าข้อความหาค่าเฉลี่ยไม่ได้ |

### 6.5 แปลงข้อความเป็นตัวเลข (Encoding)
| คอลัมน์ | วิธี | เหตุผล |
|---|---|---|
| `RainToday` (Yes/No) | Map เป็น 1/0 | มีเพียง 2 ค่า ใช้คอลัมน์เดียวเพียงพอ |
| `Location`, `WindGustDir`, `WindDir9am`, `WindDir3pm` | **One-Hot Encoding** | ไม่มีลำดับ (ทิศ N ไม่ได้ "มากกว่า" ทิศ S) |

> **ข้อควรระวัง:** หลัง One-Hot ต้องทำให้ **คอลัมน์ของ Test ตรงกับ Train** ด้วย `reindex` (กันกรณีบางหมวดมีใน Train แต่ไม่มีใน Test)

### 6.6 ปรับสเกล (Feature Scaling) ด้วย MinMaxScaler
แปลงทุกคอลัมน์ให้อยู่ช่วง 0–1 → Logistic Regression เรียนรู้ได้เร็วและเสถียรขึ้น (Tree ไม่จำเป็นต้อง Scale แต่ใช้ข้อมูลชุดเดียวกันได้ ไม่เสียหาย)

`fit_transform` กับ Train (เรียนค่า min/max) → `transform` กับ Test (ใช้ค่า min/max ของ Train)

> **ผลลัพธ์:** ข้อมูลพร้อมแล้ว: ไม่มีค่าหาย, ไม่มีข้อความ, สเกล 0–1, Train และ Test มีคอลัมน์ตรงกัน

---
# ขั้นที่ 7: โมเดลที่ 1 — Logistic Regression

### หลักการ
1. คำนวณคะแนน `z = w₁x₁ + w₂x₂ + … + b`
2. แปลงเป็นความน่าจะเป็นด้วย **Sigmoid**: `p = 1 / (1 + e⁻ᶻ)` (ได้ค่า 0–1 เสมอ)
3. ถ้า `p ≥ 0.5` → ทำนายว่า "ฝนตก" (1) มิฉะนั้นทำนายว่า "ไม่ตก" (0)

ค่า `0.5` เรียกว่า **Threshold** ปรับได้ (ดูข้อ 7.6)

### 7.1 เทรนโมเดลและทำนาย

### 7.2 Accuracy: เทียบ Train vs Test และเทียบกับ Null Accuracy

> **การตีความผล:** Train ≈ Test → **ไม่ Overfitting** และ Test (≈0.85) สูงกว่า Null accuracy (≈0.78) → โมเดลมีประโยชน์จริง

### 7.3 ทดลองปรับ Hyperparameter `C` (ความแรงของ Regularization)
- `C` เล็ก → ลงโทษ coefficient แรง → โมเดลเรียบง่าย (เสี่ยง Underfit)
- `C` ใหญ่ → ลงโทษน้อย → โมเดลซับซ้อน (เสี่ยง Overfit)

### 7.4 Confusion Matrix: รูปแบบการทำนายถูกและผิด

ใน scikit-learn: **แถว = ค่าจริง**, **คอลัมน์ = ค่าที่ทำนาย** (เรียงคลาส 0, 1)

| | ทำนาย 0 (ไม่ตก) | ทำนาย 1 (ตก) |
|---|---|---|
| **จริง 0 (ไม่ตก)** | **TN** | **FP** (เตือนฝนแต่ไม่ตก) |
| **จริง 1 (ตก)** | **FN** (ไม่เตือนแต่ฝนตก) | **TP** |

> **ข้อควรระวัง:** บางบทความ (รวมถึงต้นฉบับ) อ่านตำแหน่งสลับกัน ให้ยึดตารางนี้ หรือใช้ `cm.ravel()` ซึ่งคืนค่าเรียง `TN, FP, FN, TP` เสมอ

### 7.5 Metrics จาก Confusion Matrix

| Metric | สูตร | ความหมาย (โจทย์ฝน) |
|---|---|---|
| **Accuracy** | (TP+TN) / ทั้งหมด | ทำนายถูกกี่ % |
| **Precision** | TP / (TP+FP) | ที่บอกว่า "ฝนตก" ถูกจริงกี่ % |
| **Recall (Sensitivity)** | TP / (TP+FN) | วันฝนตกจริง ตรวจพบได้กี่ % |
| **Specificity** | TN / (TN+FP) | วันไม่ตกจริง ทำนายถูกกี่ % |
| **F1-score** | 2·P·R / (P+R) | ค่าเฉลี่ยแบบสมดุลของ Precision กับ Recall |

ใช้ `classification_report` ได้ผลเดียวกันในบรรทัดเดียว (แถว `1` คือคลาสฝนตก)

> **การตีความผล:** Precision ของฝนตกค่อนข้างดี (~0.72) แต่ **Recall เพียง ~0.51** → วันที่ฝนตกจริง โมเดลพลาดไปเกือบครึ่ง
> สาเหตุหลักคือข้อมูลไม่สมดุล โมเดลจึงเอนไปทาง "ไม่ตก" → แก้ได้ด้วยการลด Threshold

### 7.6 ปรับ Threshold: แลก Precision กับ Recall

> **หมายเหตุ:** Threshold ต่ำลง → Recall สูงขึ้น (จับวันฝนตกได้มากขึ้น) แต่ Precision ลดลง (เตือนผิดบ่อยขึ้น)
> **เลือกตามธุรกิจ:** ถ้าการพลาดวันฝนตกมีต้นทุนสูง (เช่น จัดงานกลางแจ้ง) ให้ใช้ Threshold ต่ำลง

### 7.7 ROC Curve และ AUC
- ROC = กราฟ True Positive Rate (Recall) เทียบ False Positive Rate ที่ **ทุก Threshold**
- **AUC** = พื้นที่ใต้กราฟ: 1.0 = สมบูรณ์แบบ, 0.5 = เทียบเท่าการสุ่ม (เส้นทแยง)

### 7.8 k-Fold Cross-Validation: ความเสถียรของผลลัพธ์
แบ่ง Train เป็น 5 ส่วน สลับกันเป็นชุดทดสอบ → ได้คะแนน 5 ค่า ถ้าใกล้กันแปลว่าโมเดลเสถียร

### 7.9 หา `C` ที่ดีที่สุดด้วย GridSearchCV
GridSearchCV = ทดลองทุกค่าที่กำหนด × Cross-Validation แล้วเลือกค่าที่ดีที่สุดให้อัตโนมัติ (ใช้ Train เท่านั้น)

### 7.10 การวิเคราะห์ผล Logistic Regression

| ประเด็น | ผลลัพธ์ | การตีความ |
|---|---|---|
| ดีกว่า Baseline หรือไม่ | Test Acc **0.846** vs Null **0.776** | สูงกว่าการทำนายว่า "ไม่ตก" ทุกวันประมาณ 7 จุด (ส่วนต่างน้อยกว่าที่ตัวเลข 85% อาจทำให้เข้าใจ) |
| Overfit หรือไม่ | Train 0.849 vs Test 0.846 | แทบเท่ากัน → ไม่ Overfit |
| เสถียรหรือไม่ | CV 0.848 ± 0.003 | คะแนน 5 รอบแทบเท่ากัน |
| บอกว่า "ฝนตก" เชื่อได้เพียงใด? | Precision **0.72** | บอกว่าตก 4 ครั้ง ถูก ~3 ครั้ง |
| วันฝนตกจริงตรวจพบได้เพียงใด? | Recall **0.51** | จุดอ่อนหลัก: จากวันฝนตกจริง 6,375 วัน ตรวจพบ 3,242 / **พลาด 3,133 วัน** |
| วันไม่ตกทำนายถูกเพียงใด? | Specificity 0.94 | เตือนผิด (FP) เพียง 1,235 จาก 22,064 วัน |
| แยกสองคลาสได้ดีเพียงใด? | ROC-AUC **0.868** | ระดับดี (0.5 = เทียบเท่าการสุ่ม, 1.0 = สมบูรณ์แบบ) |
| `C` มีผลหรือไม่ | C = 0.01 → 0.840, C = 1 หรือ 100 → 0.846 | แทบไม่มีผล — โมเดลไม่ได้ Overfit จึงไม่ต้องพึ่ง Regularization มาก |
| ปรับ Threshold ช่วยหรือไม่ | Threshold 0.3 → Recall 0.69, Precision 0.59, **F1 0.64** (สูงสุด) | แลก Precision เล็กน้อย เพื่อจับวันฝนตกได้เพิ่ม ~18 จุด |

> **สรุป:** โมเดลเสถียรและไม่ Overfit, Precision ดี แต่ **Recall ต่ำเพราะข้อมูลไม่สมดุล** (ฝนตกเพียง 22%)
> ถ้าการพลาดวันฝนตกมีความเสียหายสูง ควรใช้ Threshold ≈ 0.3 แทน 0.5

---
# ขั้นที่ 8: โมเดลที่ 2 — Decision Tree

### หลักการ
สร้าง "ต้นไม้คำถาม Yes/No" เช่น *ความชื้นบ่าย > 70%?* → *แดด < 5 ชม.?* → ทำนาย
แต่ละจุดแยกเลือกคำถามที่ทำให้กลุ่มลูก **บริสุทธิ์ที่สุด** (Gini impurity ต่ำสุด)

**จุดอ่อน:** ถ้าไม่จำกัดความลึก ต้นไม้จะ "จำ" ข้อมูล Train ได้หมด → **Overfitting**

### 8.1 ทดลอง: ไม่จำกัดความลึก

### 8.2 การเลือกความลึกของต้นไม้ (Train vs Test)

> **ข้อสังเกต:** ยิ่งลึก Train ยิ่งสูง แต่ Test ขึ้นถึงจุดหนึ่ง (~8) แล้ว **ลดลง** = เริ่ม Overfit

### 8.3 เลือก `max_depth` อย่างถูกวิธีด้วย GridSearchCV
(ไม่ควรเลือกจากกราฟ Test ข้างบน เพราะเท่ากับใช้ Test ในการตัดสินใจ)

### 8.4 ดูหน้าตาต้นไม้ (3 ชั้นแรก) และ Feature Importance

> **ข้อสังเกต:** `Humidity3pm` สำคัญที่สุด สอดคล้องกับที่เห็นใน EDA (ข้อ 5.4)

### 8.5 ผลบน Test Set

### 8.6 การวิเคราะห์ผล Decision Tree

| ประเด็น | ผลลัพธ์ | การตีความ |
|---|---|---|
| ไม่จำกัดความลึกเป็นอย่างไร? | ลึก 42 ชั้น, Train **1.000** vs Test **0.790** | Overfit หนัก — จำข้อมูล Train ได้หมด และสูงกว่า Null accuracy (0.776) เพียงเล็กน้อย |
| หลังจูนด้วย GridSearchCV | `max_depth=10`, `min_samples_leaf=50` → Train 0.854 vs Test **0.837** | ช่องว่างเหลือ ~1.6 จุด แก้ Overfit ได้ |
| Precision / Recall / F1 (ฝนตก) | 0.70 / 0.49 / 0.57 | Recall ต่ำแบบเดียวกับ Logistic Regression และต่ำกว่าเล็กน้อยทุกตัว |
| ROC-AUC | **0.850** | ต่ำสุดในสามโมเดล |
| ใช้ Feature อะไรตัดสิน? | `Humidity3pm` สำคัญ ~0.63 (เกินครึ่ง) | ต้นไม้ต้นเดียวพึ่ง Feature เดียวเป็นหลัก |

> **สรุป:** แม่นน้อยที่สุดในสามโมเดล แต่ **ตีความง่ายที่สุด** — วาดเป็นกฎ "ถ้า...แล้ว..." ให้คนทั่วไปอ่านได้
> และเป็นตัวอย่างชัดเจนที่สุดว่า **การจำกัดความลึกสำคัญมาก** กับโมเดล Tree

---
# ขั้นที่ 9: โมเดลที่ 3 — Random Forest

### หลักการ
สร้าง Decision Tree **หลายร้อยต้น** แต่ละต้นเห็นข้อมูลและ Feature ต่างส่วนกัน (สุ่ม) แล้วให้ **โหวต** คำตอบ
→ ความผิดพลาดของแต่ละต้นหักล้างกัน ได้ผลแม่นและเสถียรกว่าต้นไม้ต้นเดียว (ลด Overfitting)

| Hyperparameter | ความหมาย | ค่าที่ใช้ |
|---|---|---|
| `n_estimators` | จำนวนต้นไม้ | 100 |
| `max_depth` | ความลึกสูงสุดของแต่ละต้น | 20 (กันไม่ให้ต้นไม้ใหญ่เกินไป) |
| `n_jobs=-1` | ใช้ CPU ทุก core | เร็วขึ้น |

> **ข้อสังเกต:** Train สูงกว่า Test พอสมควร (Overfit บ้าง) แต่ **Test ยังสูงกว่า Decision Tree ต้นเดียว** — นี่คือจุดเด่นของ Ensemble

### 9.1 ผลบน Test Set

### 9.2 การวิเคราะห์ผล Random Forest

| ประเด็น | ผลลัพธ์ | การตีความ |
|---|---|---|
| แม่นเพียงใด? | Test Acc **0.853** | สูงสุดในสามโมเดล |
| Overfit หรือไม่ | Train 0.943 vs Test 0.853 | ช่องว่าง ~9 จุด Overfit บ้าง แต่ Test ยังดีที่สุด (แต่ละต้นลึกได้ถึง 20 ชั้น) |
| บอกว่า "ฝนตก" เชื่อได้เพียงใด? | Precision **0.78** | สูงสุด — เตือนผิดน้อยที่สุด |
| วันฝนตกจริงตรวจพบได้เพียงใด? | Recall **0.48** | ต่ำสุดเล็กน้อย — โมเดลมีแนวโน้มทำนายว่าฝนตกเฉพาะเมื่อมีความมั่นใจสูง |
| F1 | 0.595 | ใกล้เคียง Logistic Regression (0.597) |
| ROC-AUC | **0.878** | สูงสุด — แยกวันฝนตก/ไม่ตกได้ดีที่สุด |
| ใช้ Feature อะไรตัดสิน? | `Humidity3pm` ~0.17 ที่เหลือกระจาย ~0.04–0.05 | ใช้ข้อมูลหลายตัวร่วมกัน ไม่พึ่งตัวเดียวแบบ Decision Tree |

> **สรุป:** แม่นที่สุดทั้ง Accuracy, Precision และ AUC เพราะเฉลี่ยคำตอบจากต้นไม้ 100 ต้น
> แต่ Recall ยังต่ำ — ทดลองลด Threshold แบบข้อ 7.6 ได้เช่นกัน (`rf.predict_proba(X_test)[:, 1] >= 0.3`)

---
# ขั้นที่ 10: เปรียบเทียบ 3 โมเดลและเลือกโมเดลที่ดีที่สุด

### 10.1 ตารางเปรียบเทียบ Metrics (บน Test Set)

### 10.2 ROC Curve ของทั้ง 3 โมเดล

### 10.3 Confusion Matrix ของทั้ง 3 โมเดล

### 10.4 สรุปผลการเปรียบเทียบ

| โมเดล | Test Acc | Precision | Recall | F1 | ROC-AUC | Overfit? | จุดเด่น |
|---|---|---|---|---|---|---|---|
| Logistic Regression | 0.846 | 0.72 | **0.51** | **0.597** | 0.868 | ไม่ | เสถียร เร็ว อธิบายง่าย, F1/Recall สูงสุด |
| Decision Tree | 0.837 | 0.70 | 0.49 | 0.574 | 0.850 | ไม่ (หลังปรับ Hyperparameter) | ตีความง่ายที่สุด |
| Random Forest | **0.853** | **0.78** | 0.48 | 0.595 | **0.878** | เล็กน้อย | แม่นที่สุด, AUC สูงสุด |

**ข้อค้นพบหลัก**
1. **ทุกโมเดลให้ผลดีกว่า Baseline** (Null accuracy 0.776) แต่ส่วนต่างเพียง 6–8 จุด → ต้องรายงาน Null accuracy ด้วยทุกครั้ง
2. **ทั้งสามโมเดลแม่นใกล้เคียงกันมาก** (Accuracy ห่างกันไม่ถึง 2 จุด) โมเดลที่ซับซ้อนขึ้นไม่ได้ช่วยมาก
3. **จุดอ่อนร่วม = Recall ≈ 0.5** มาจากข้อมูลไม่สมดุล ไม่ใช่จากโมเดล → แก้ด้วยการลด Threshold หรือ `class_weight='balanced'`
4. **ความชื้นตอนบ่าย (`Humidity3pm`)** เป็นตัวทำนายที่สำคัญที่สุดในทุกโมเดล ตรงกับ EDA

**คำแนะนำ**
- ต้องการความแม่นสูงสุด → **Random Forest**
- ต้องการโมเดลเรียบง่าย อธิบายได้ ใช้ทรัพยากรน้อย → **Logistic Regression** (เสียความแม่นไปเพียง ~0.7 จุด)
- ใช้จริงควรปรับ Threshold ≈ 0.3 เพื่อให้จับวันฝนตกได้มากขึ้น

> **ตัวอย่างการเขียนสรุปผลในข้อสอบ:**
> *"เลือก Random Forest เพราะได้ Accuracy (0.853) และ ROC-AUC (0.878) สูงสุดบน Test Set และสูงกว่า Null accuracy (0.776)
> แม้ Train/Test ต่างกัน ~9 จุดแต่ Test ยังดีที่สุด ข้อจำกัดคือ Recall ของวันฝนตกเพียง 0.48 เนื่องจากข้อมูลไม่สมดุล
> หากต้องการลดการพลาดวันฝนตก ควรลด Threshold หรือใช้ class_weight='balanced'"*
>
> โครงที่ใช้: (1) เลือกโมเดลอะไร (2) ตัวเลขที่ใช้ตัดสิน (3) เหตุผล (4) ข้อจำกัดและสิ่งที่จะปรับต่อ

---
## รายการตรวจสอบสำหรับการสอบ (Checklist)
- [ ] ลบแถวที่ Target หาย / ตรวจข้อมูลซ้ำ
- [ ] ดูสัดส่วน Target และคำนวณ Null accuracy
- [ ] **Split ก่อน** แล้วค่อยจัด Outlier / เติมค่าหาย / Scale ด้วยค่าจาก Train
- [ ] Encode ข้อความ และจัดคอลัมน์ Test ให้ตรง Train
- [ ] เทรนอย่างน้อย 2–3 โมเดล เทียบ Train vs Test (เช็ค Overfit)
- [ ] รายงาน Confusion Matrix, Precision, Recall, F1, ROC-AUC
- [ ] ใช้ Cross-Validation / GridSearchCV เลือก Hyperparameter (ไม่ใช้ Test)
- [ ] สรุปเลือกโมเดลพร้อมเหตุผล

## แบบฝึกหัด
1. ทดลองเพิ่ม `class_weight='balanced'` ให้ Logistic Regression แล้วดู Recall เปลี่ยนไปอย่างไร
2. เพิ่มโมเดลที่ 4: `KNeighborsClassifier(n_neighbors=15)` แล้วเพิ่มลงในตารางเปรียบเทียบ